## 1 

In [6]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import random
import unicodedata
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from transformers import XLMRobertaModel, XLMRobertaConfig, XLMRobertaForCausalLM
import timm
import Levenshtein
from tqdm import tqdm
import cv2
import numpy as np
from sklearn.model_selection import train_test_split

# ============================================================
# 1. HINDI PATHS & CONFIGURATIONS (From Hindi mGPT Script)
# ============================================================
ROOT_DIR        = "/home/mca24-26/ocr/dataset/Word_Level_Hindi_Training_Set/Word_Level_Training_Set"
TRAIN_TXT       = os.path.join(ROOT_DIR, "train.txt")
CHECKPOINT_PATH = "/home/mca24-26/ocr/hindi_xlm.pth"

IMG_HEIGHT   = 64   
IMG_WIDTH    = 256   
MAX_SEQ_LEN  = 40
BEAM_SIZE    = 5
D_MODEL      = 768
SEED         = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed(SEED)

# ============================================================
# 2. HINDI VOCABULARY & TOKENIZER MECHANICS
# ============================================================
HINDI_CHARS = (
    "अआइईउऊऋएऐओऔ"
    "कखगघङचछजझञ"
    "टठडढणतथदधन"
    "पफबभमयरलव"
    "शषसह"
    "ािीुूृेैोौंःॅ्"
    "०१२३४५६७८९"
    "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ"
    ".,!?()[]{}:;-_'/\\&%$#@+*= "
)

SPECIAL_TOKENS = ["<PAD>", "<SOS>", "<EOS>", "<UNK>"]
ALL_TOKENS = SPECIAL_TOKENS + list(HINDI_CHARS)
char2idx = {c: i for i, c in enumerate(ALL_TOKENS)}
idx2char = {i: c for c, i in char2idx.items()}

VOCAB_SIZE = len(ALL_TOKENS)
PAD_IDX = char2idx["<PAD>"]
SOS_IDX = char2idx["<SOS>"]
EOS_IDX = char2idx["<EOS>"]
UNK_IDX = char2idx["<UNK>"]

# Tokenizer wrapper class to maintain API compatibility with the training loops
class HindiCustomTokenizer:
    def __init__(self):
        self.vocab_size = VOCAB_SIZE
        self.pad_token_id = PAD_IDX
        self.bos_token_id = SOS_IDX
        self.eos_token_id = EOS_IDX
        self.unk_token_id = UNK_IDX

    def encode(self, text):
        tokens = [self.bos_token_id]
        for ch in text:
            tokens.append(char2idx.get(ch, self.unk_token_id))
        tokens.append(self.eos_token_id)
        if len(tokens) < MAX_SEQ_LEN:
            tokens += [self.pad_token_id] * (MAX_SEQ_LEN - len(tokens))
        else:
            tokens = tokens[:MAX_SEQ_LEN]
            tokens[-1] = self.eos_token_id
        return torch.tensor(tokens, dtype=torch.long)

    def decode(self, tokens, skip_special_tokens=True):
        chars = []
        for t in tokens:
            t = int(t)
            if t == self.eos_token_id:
                break
            if skip_special_tokens and t in [self.pad_token_id, self.bos_token_id]:
                continue
            chars.append(idx2char.get(t, ""))
        return "".join(chars)

# ============================================================
# 3. IMAGE PREPROCESSING (From Hindi mGPT Script)
# ============================================================
def preprocess_image(img_path):
    img = cv2.imread(img_path)
    if img is None:
        # Fallback for broken/missing images
        return torch.ones((3, IMG_HEIGHT, IMG_WIDTH), dtype=torch.float32) * -1.0
        
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Deskew
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
    edges = cv2.Canny(thresh, 50, 150, apertureSize=3)
    lines = cv2.HoughLinesP(edges, 1, np.pi/180, threshold=40, minLineLength=30, maxLineGap=5)
    angle = 0.0
    if lines is not None:
        angles = []
        for line in lines:
            x1, y1, x2, y2 = line[0]
            if x2 != x1:
                angles.append(np.arctan2(y2 - y1, x2 - x1) * 180 / np.pi)
        if angles:
            angle = np.median(angles)
    if abs(angle) > 20:
        angle = 0.0
    (h, w) = img.shape[:2]
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    deskewed = cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)
    
    # Resize + pad aspect ratio safely
    h, w = deskewed.shape[:2]
    scale = IMG_HEIGHT / max(h, 1)
    new_w = int(w * scale)
    resized = cv2.resize(deskewed, (new_w, IMG_HEIGHT))
    if new_w < IMG_WIDTH:
        pad = np.ones((IMG_HEIGHT, IMG_WIDTH - new_w, 3), dtype=np.uint8) * 255
        resized = np.concatenate([resized, pad], axis=1)
    else:
        resized = cv2.resize(resized, (IMG_WIDTH, IMG_HEIGHT))
    
    # Normalize values to PyTorch Tensors
    img_tensor = resized.astype(np.float32) / 255.0
    img_tensor = (img_tensor - 0.5) / 0.5
    img_tensor = np.transpose(img_tensor, (2, 0, 1))
    return torch.tensor(img_tensor, dtype=torch.float32)

# ============================================================
# 4. DATASET & DATALOADERS CREATION
# ============================================================
class HindiWordDataset(Dataset):
    def __init__(self, samples, tokenizer):
        self.samples = samples
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, text = self.samples[idx]
        image = preprocess_image(img_path)
        labels = self.tokenizer.encode(text)
        
        # Cross-entropy mapping masking ignore index (-100)
        labels = torch.where(labels == self.tokenizer.pad_token_id, torch.tensor(-100), labels)
        return image, labels, text

def get_hindi_dataloaders(root_dir, train_txt_path, tokenizer, batch_size=32):
    samples = []
    with open(train_txt_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split("\t")
            if len(parts) != 2:
                continue
            rel_path, text = parts
            text = unicodedata.normalize("NFC", text.strip())
            img_path = os.path.join(root_dir, rel_path)
            if os.path.exists(img_path):
                samples.append((img_path, text))

    print(f"Total Registered Hindi Samples: {len(samples)}")
    
    train_samples, val_samples = train_test_split(samples, test_size=0.15, random_state=SEED)
    
    train_dataset = HindiWordDataset(train_samples, tokenizer)
    val_dataset = HindiWordDataset(val_samples, tokenizer)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
    
    return train_loader, val_loader

# ============================================================
# 5. ARCHITECTURE PIECES (Unchanged pipeline models)
# ============================================================
class LocalizationNetwork(nn.Module):
    def __init__(self, num_control_points=16):
        super().__init__()
        self.num_control_points = num_control_points
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(True), nn.AdaptiveAvgPool2d((4, 4))
        )
        self.fc = nn.Sequential(
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(True),
            nn.Linear(256, num_control_points * 2)
        )
        self.fc[-1].weight.data.zero_()
        self.fc[-1].bias.data.zero_()

    def forward(self, x):
        B = x.size(0)
        pts = self.fc(self.conv(x).view(B, -1))
        return pts.view(B, self.num_control_points, 2)


class TPSSpatialTransformer(nn.Module):
    def __init__(self, num_control_points=16):
        super().__init__()
        self.num_control_points = num_control_points
        self.localization = LocalizationNetwork(num_control_points)
        
        r1 = torch.linspace(-0.9, 0.9, num_control_points // 2)
        r2 = torch.tensor([-0.7, 0.7])
        grid_Y, grid_X = torch.meshgrid(r2, r1, indexing='ij')
        target_control_points = torch.stack([grid_X.flatten(), grid_Y.flatten()], dim=1) 
        self.register_buffer('target_control_points', target_control_points)
        
    def _compute_partial_solutions(self, X, Y):
        N = X.size(0)
        M = Y.size(0)
        X_tiled = X.unsqueeze(1).repeat(1, M, 1)
        Y_tiled = Y.unsqueeze(0).repeat(N, 1, 1)
        dist_sq = torch.sum((X_tiled - Y_tiled) ** 2, dim=2)
        dist_sq[dist_sq == 0] = 1.0
        return dist_sq * torch.log(dist_sq)

    def forward(self, x):
        B, C, H, W = x.size()
        source_control_points = self.localization(x) + self.target_control_points.unsqueeze(0)
        
        N = self.num_control_points
        L = torch.zeros(B, N + 3, N + 3, device=x.device)
        K = self._compute_partial_solutions(self.target_control_points, self.target_control_points)
        L[:, :N, :N] = K.unsqueeze(0)
        P = torch.cat([torch.ones(N, 1, device=x.device), self.target_control_points], dim=1)
        L[:, :N, N:] = P.unsqueeze(0)
        L[:, N:, :N] = P.t().unsqueeze(0)
        
        Y = torch.zeros(B, N + 3, 2, device=x.device)
        Y[:, :N, :] = source_control_points
        
        W_coef = torch.linalg.solve(L, Y)
        
        grid_Y, grid_X = torch.meshgrid(torch.linspace(-1, 1, H), torch.linspace(-1, 1, W), indexing='ij')
        eval_points = torch.stack([grid_X.flatten(), grid_Y.flatten()], dim=1).to(x.device) 
        
        K_eval = self._compute_partial_solutions(eval_points, self.target_control_points) 
        P_eval = torch.cat([torch.ones(H * W, 1, device=x.device), eval_points], dim=1) 
        
        transformed_points = torch.matmul(K_eval, W_coef[:, :N, :]) + torch.matmul(P_eval, W_coef[:, N:, :])
        grid = transformed_points.view(B, H, W, 2)
        
        return F.grid_sample(x, grid, align_corners=False, padding_mode='border')


class SwinEncoder(nn.Module):
    def __init__(self, d_model=D_MODEL):
        super().__init__()
        self.swin = timm.create_model(
            "swin_base_patch4_window7_224",
            pretrained=True,
            features_only=True,
            img_size=(IMG_HEIGHT, IMG_WIDTH),
            strict_img_size=False,
        )
        self.proj = nn.Linear(512, d_model)  
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        features   = self.swin(x)
        feat       = features[-2]          
        B, H, W, C = feat.shape            
        feat       = feat.view(B, H*W, C)  
        return self.norm(self.proj(feat))


class XLMRobertaDecoder(nn.Module):
    def __init__(self, vocab_size, d_model=D_MODEL):
        super().__init__()
        xlm_roberta = XLMRobertaModel.from_pretrained("xlm-roberta-base")
        config      = XLMRobertaConfig.from_pretrained("xlm-roberta-base")
        config.is_decoder          = True
        config.add_cross_attention = True
        config.vocab_size          = vocab_size
        self.decoder = XLMRobertaForCausalLM(config)
        
        # Extract and slice the embedding weights to match your small vocabulary size
        emb_state_dict = xlm_roberta.embeddings.state_dict()
        if "word_embeddings.weight" in emb_state_dict:
            emb_state_dict["word_embeddings.weight"] = emb_state_dict["word_embeddings.weight"][:vocab_size]
            
        # Safely load the sliced embeddings
        self.decoder.roberta.embeddings.load_state_dict(emb_state_dict, strict=False)
        
        # Load the remaining transformer layers unchanged
        for i, layer in enumerate(self.decoder.roberta.encoder.layer[:6]):
            layer.load_state_dict(xlm_roberta.encoder.layer[i].state_dict(), strict=False)
        del xlm_roberta

    def forward(self, input_ids, encoder_hidden_states=None, labels=None):
        return self.decoder(
            input_ids=input_ids, encoder_hidden_states=encoder_hidden_states, labels=labels
        )


class CompleteHTRPipeline(nn.Module):
    def __init__(self, vocab_size, d_model=D_MODEL, num_control_points=16):
        super().__init__()
        self.tps_stn         = TPSSpatialTransformer(num_control_points)
        self.swin_encoder    = SwinEncoder(d_model=d_model)
        self.bilstm          = nn.LSTM(
            input_size=d_model, hidden_size=d_model // 2, num_layers=2,
            batch_first=True, bidirectional=True, dropout=0.3
        )
        self.roberta_decoder = XLMRobertaDecoder(vocab_size=vocab_size, d_model=d_model)

    def _extract_visual_memory(self, images):
        rectified = self.tps_stn(images)
        swin_feat = self.swin_encoder(rectified)
        memory, _ = self.bilstm(swin_feat)
        return memory.contiguous()

    def forward(self, images, target_ids, criterion=None):
        memory = self._extract_visual_memory(images)
        
        dec_input = target_ids[:, :-1].clone()
        labels = target_ids[:, 1:].clone()
        
        dec_input[dec_input == -100] = PAD_IDX 
        
        outputs = self.roberta_decoder(input_ids=dec_input, encoder_hidden_states=memory)
        logits = outputs.logits

        if criterion is not None:
            return criterion(logits.reshape(-1, logits.size(-1)), labels.reshape(-1))
        return F.cross_entropy(logits.reshape(-1, logits.size(-1)), labels.reshape(-1), ignore_index=-100)

    @torch.no_grad()
    def generate_greedy(self, images, max_length=MAX_SEQ_LEN, bos_token_id=0, eos_token_id=2):
        device = images.device
        B = images.size(0)
        memory = self._extract_visual_memory(images)
        
        generated = torch.full((B, 1), bos_token_id, dtype=torch.long, device=device)
        finished = torch.zeros(B, dtype=torch.bool, device=device)
        
        total_probs = torch.zeros(B, device=device)
        seq_lengths = torch.zeros(B, device=device)
    
        for _ in range(max_length - 1):
            outputs = self.roberta_decoder(input_ids=generated, encoder_hidden_states=memory)
            next_token_logits = outputs.logits[:, -1, :]
            
            probs = F.softmax(next_token_logits, dim=-1)
            next_token_probs, next_tokens = probs.max(dim=-1, keepdim=True)
            generated = torch.cat([generated, next_tokens], dim=1)
            
            active_mask = ~finished
            total_probs += next_token_probs.squeeze(-1) * active_mask
            seq_lengths += active_mask.float()
            
            finished |= (next_tokens.squeeze(-1) == eos_token_id)
            if finished.all():
                break
    
        seq_lengths = torch.clamp(seq_lengths, min=1.0)
        confidence_scores = (total_probs / seq_lengths).cpu().numpy()
        
        return generated[:, 1:], confidence_scores


# ============================================================
# 6. AGENTIC VERIFICATION MODULE
# ============================================================
class AgenticVerificationModule:
    def __init__(self, train_txt_file=None):
        self.lexicon   = {}   
        self.freq_max  = 1
        if train_txt_file and os.path.exists(train_txt_file):
            self._build_lexicon(train_txt_file)

    def _build_lexicon(self, file_path):
        freq = {}
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                parts = line.split("\t")
                if len(parts) == 2:
                    word = unicodedata.normalize("NFC", parts[1].strip())
                    if word:
                        freq[word] = freq.get(word, 0) + 1
        self.lexicon  = freq
        self.freq_max = max(freq.values()) if freq else 1
        print(f"Lexicon built: {len(self.lexicon)} unique Hindi words.")

    def _normalized_freq(self, word):
        return self.lexicon.get(word, 0) / self.freq_max

    def verify_and_correct(self, text_output, confidence=None, confidence_threshold=0.85):
        cleaned = text_output.strip()

        if (not cleaned
                or cleaned in self.lexicon
                or len(cleaned) <= 1):
            return cleaned

        if confidence is not None and confidence >= confidence_threshold:
            return cleaned

        best_candidate = None
        best_score     = -float('inf')
        target_len     = len(cleaned)

        for word in self.lexicon:
            if abs(len(word) - target_len) > 2:
                continue
            
            dist = Levenshtein.distance(cleaned, word)
            if dist > 2: 
                continue

            freq_score = self._normalized_freq(word)
            score      = freq_score - (dist * 1.2)

            if score > best_score:
                best_score     = score
                best_candidate = word

        if best_candidate is None:
            return cleaned

        return best_candidate


# ============================================================
# 7. METRICS & EARLY STOPPING
# ============================================================
def calculate_metrics(predictions, targets):
    if not targets:
        return {"CER": 0.0, "WER": 0.0}
    total_cer   = 0.0
    wrong_words = 0
    for pred, target in zip(predictions, targets):
        p = pred.strip()
        t = target.strip()
        d = Levenshtein.distance(p, t)
        if len(t) > 0:
            total_cer += d / len(t)
        elif len(p) > 0:
            total_cer += 1.0
        if d != 0:
            wrong_words += 1
    n = len(targets)
    return {
        "CER": round((total_cer / n) * 100, 4),
        "WER": round((wrong_words / n) * 100, 4)
    }


class EarlyStopping:
    def __init__(self, patience=8, min_delta=0.05):
        self.patience   = patience
        self.min_delta  = min_delta
        self.counter    = 0
        self.best       = float('inf')
        self.early_stop = False

    def __call__(self, metric):
        if metric < self.best - self.min_delta:
            self.best    = metric
            self.counter = 0
        else:
            self.counter += 1
            print(f"EarlyStopping: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        return self.early_stop


# ============================================================
# 8. LLRD OPTIMIZER BUILDER
# ============================================================
def build_llrd_optimizer(model, base_lr=5e-5, decay_factor=0.75, weight_decay=0.05):
    lr0 = base_lr
    lr1 = base_lr * decay_factor
    lr2 = base_lr * (decay_factor ** 2)
    lr3 = base_lr * (decay_factor ** 3)
    lr4 = base_lr * (decay_factor ** 4)
    lr5 = base_lr * (decay_factor ** 5)
    lr6 = base_lr * (decay_factor ** 6)

    assigned = set()
    decoder_params = []
    for name, param in model.roberta_decoder.named_parameters():
        if id(param) not in assigned:
            decoder_params.append(param)
            assigned.add(id(param))

    bilstm_params = []
    for name, param in model.bilstm.named_parameters():
        if id(param) not in assigned:
            bilstm_params.append(param)
            assigned.add(id(param))

    swin_proj_params = []
    for name, param in model.swin_encoder.named_parameters():
        if id(param) not in assigned and (name.startswith("proj.") or name.startswith("norm.")):
            swin_proj_params.append(param)
            assigned.add(id(param))

    swin_s4_params = []
    for name, param in model.swin_encoder.swin.named_parameters():
        if id(param) not in assigned and "layers_3" in name:
            swin_s4_params.append(param)
            assigned.add(id(param))

    swin_s3_params = []
    for name, param in model.swin_encoder.swin.named_parameters():
        if id(param) not in assigned and "layers_2" in name:
            swin_s3_params.append(param)
            assigned.add(id(param))

    swin_s2_params = []
    for name, param in model.swin_encoder.swin.named_parameters():
        if id(param) not in assigned and "layers_1" in name:
            swin_s2_params.append(param)
            assigned.add(id(param))

    swin_s1_params = []
    for name, param in model.swin_encoder.swin.named_parameters():
        if id(param) not in assigned:
            swin_s1_params.append(param)
            assigned.add(id(param))

    tps_params = []
    for name, param in model.tps_stn.named_parameters():
        if id(param) not in assigned:
            tps_params.append(param)
            assigned.add(id(param))

    param_groups = [
        {"params": decoder_params,   "lr": lr0, "name": "xlm_roberta_decoder"},
        {"params": bilstm_params,    "lr": lr1, "name": "bilstm"},
        {"params": swin_proj_params, "lr": lr2, "name": "swin_proj"},
        {"params": swin_s4_params,   "lr": lr3, "name": "swin_stage4"},
        {"params": swin_s3_params,   "lr": lr4, "name": "swin_stage3"},
        {"params": swin_s2_params,   "lr": lr5, "name": "swin_stage2"},
        {"params": swin_s1_params,   "lr": lr6, "name": "swin_stage1_embed"},
        {"params": tps_params,       "lr": lr2, "name": "tps_stn"},
    ]

    param_groups = [g for g in param_groups if len(g["params"]) > 0]

    print("\nLLRD Optimizer Groups:")
    print(f"{'Group':<22} {'LR':>10} {'Params':>10}")
    print("-" * 47)
    for g in param_groups:
        n = sum(p.numel() for p in g["params"])
        print(f"{g['name']:<22} {g['lr']:>10.2e} {n/1e6:>9.2f}M")

    optimizer = torch.optim.AdamW(param_groups, weight_decay=0.05)
    return optimizer


# ============================================================
# 9. RUN PIPELINE LOOP
# ============================================================
def run_training_pipeline(epochs=50, batch_size=16, base_lr=5e-5, resume_from=None):
    print("Loading custom Hindi Vocabulary Tokenizer...")
    tokenizer = HindiCustomTokenizer()
    print(f"BOS={tokenizer.bos_token_id} | EOS={tokenizer.eos_token_id} | PAD={tokenizer.pad_token_id}")

    train_loader, val_loader = get_hindi_dataloaders(
        ROOT_DIR, TRAIN_TXT, tokenizer, batch_size=batch_size
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model  = CompleteHTRPipeline(vocab_size=tokenizer.vocab_size).to(device)

    total_p     = sum(p.numel() for p in model.parameters())
    trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total params    : {total_p/1e6:.2f}M")
    print(f"Trainable params: {trainable_p/1e6:.2f}M")

    criterion = nn.CrossEntropyLoss(ignore_index=-100, label_smoothing=0.1)
    optimizer = build_llrd_optimizer(model, base_lr=base_lr, decay_factor=0.75, weight_decay=0.05)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-7)

    agent_verifier = AgenticVerificationModule(train_txt_file=TRAIN_TXT)
    early_stopper  = EarlyStopping(patience=12, min_delta=0.1)
    best_val_wer   = float('inf')
    start_epoch    = 1

    if resume_from and os.path.exists(resume_from):
        print(f"\nResuming from: {resume_from}")
        ckpt = torch.load(resume_from, map_location=device)
        model.load_state_dict(ckpt['model_state_dict'])
        best_val_wer = ckpt.get('best_wer', float('inf'))
        start_epoch  = ckpt.get('epoch', 1) + 1
        print(f"Resumed at epoch {start_epoch} | Best WER: {best_val_wer:.2f}%")

    print(f"\nTraining on: {device}")

    for epoch in range(start_epoch, epochs + 1):
        model.train()
        total_loss = 0.0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs} [Train]")

        for images, labels, _ in pbar:
            images = images.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            loss = model(images, target_ids=labels, criterion=criterion)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item()
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})

        scheduler.step()
        avg_loss = total_loss / len(train_loader)
        lrs = [f"{g['lr']:.2e}" for g in optimizer.param_groups]
        print(f"Epoch {epoch} | Loss: {avg_loss:.4f} | LRs: decoder={lrs[0]} bilstm={lrs[1]}")

        model.eval()
        all_preds, all_targets = [], []
        
        with torch.no_grad():
            for images, _, text_labels in tqdm(val_loader, desc=f"Epoch {epoch}/{epochs} [Val]"):
                images = images.to(device)
                tokens, confidences = model.generate_greedy(
                    images,
                    max_length=MAX_SEQ_LEN,
                    bos_token_id=tokenizer.bos_token_id,
                    eos_token_id=tokenizer.eos_token_id
                )
        
                for i in range(images.size(0)):
                    raw = tokenizer.decode(tokens[i], skip_special_tokens=True)
                    
                    verified = agent_verifier.verify_and_correct(
                        raw, 
                        confidence=confidences[i],
                        confidence_threshold=0.85 
                    )                    
                    all_preds.append(verified)
                    all_targets.append(text_labels[i].strip())

        metrics     = calculate_metrics(all_preds, all_targets)
        current_wer = metrics['WER']
        print(f"Epoch {epoch} | CER: {metrics['CER']:.2f}% | WER: {current_wer:.2f}%")

        if current_wer < best_val_wer:
            best_val_wer = current_wer
            print(f"New Best WER achieved! Saving configuration parameters...")
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_wer': best_val_wer,
            }, CHECKPOINT_PATH)

        if early_stopper(current_wer):
            print("Early stopping triggered. Halting run execution.")
            break

if __name__ == "__main__":
    run_training_pipeline(epochs=50, batch_size=32, resume_from=None)

Loading custom Hindi Vocabulary Tokenizer...
BOS=1 | EOS=2 | PAD=0
Total Registered Hindi Samples: 85585
Total params    : 209.35M
Trainable params: 209.35M

LLRD Optimizer Groups:
Group                          LR     Params
-----------------------------------------------
xlm_roberta_decoder      5.00e-05    114.53M
bilstm                   3.75e-05      7.09M
swin_proj                2.81e-05      0.40M
swin_stage4              2.11e-05     27.30M
swin_stage3              1.58e-05     57.30M
swin_stage2              1.19e-05      1.71M
swin_stage1_embed        8.90e-06      0.40M
tps_stn                  2.81e-05      0.63M
Lexicon built: 7617 unique Hindi words.

Training on: cuda


Epoch 1/50 [Train]: 100%|████████████████| 2273/2273 [08:11<00:00,  4.62it/s, loss=1.1511]


Epoch 1 | Loss: 2.0063 | LRs: decoder=5.00e-05 bilstm=3.75e-05


Epoch 1/50 [Val]: 100%|█████████████████████████████████| 402/402 [01:14<00:00,  5.42it/s]


Epoch 1 | CER: 20.72% | WER: 27.89%
New Best WER achieved! Saving configuration parameters...


...

Epoch 49 | CER: 5.99% | WER: 5.06%
New Best WER achieved! Saving configuration parameters...
EarlyStopping: 5/12


Epoch 50/50 [Train]: 100%|███████████████| 2273/2273 [15:47<00:00,  2.40it/s, loss=0.8209]


Epoch 50 | Loss: 0.8211 | LRs: decoder=1.00e-07 bilstm=1.00e-07


Epoch 50/50 [Val]: 100%|████████████████████████████████| 402/402 [02:05<00:00,  3.20it/s]

Epoch 50 | CER: 6.01% | WER: 5.07%
EarlyStopping: 6/12


In [ ]:
# =====================================================================
# TEST SCRIPT FOR HINDI HTR – XLM‑RoBERTa PIPELINE
# (Swin + TPS + BiLSTM + XLM‑RoBERTa Decoder + LLRD + Agentic Verification)
# =====================================================================

import os
import cv2
import random
import unicodedata
import numpy as np
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torch.optim import AdamW
from torch.amp import autocast, GradScaler

import timm
from transformers import XLMRobertaModel, XLMRobertaConfig, XLMRobertaForCausalLM

import Levenshtein
from jiwer import wer
from tqdm import tqdm
from sklearn.model_selection import train_test_split

# ============================================================
# CONFIGURATION (must match training)
# ============================================================

ROOT_DIR = "/home/mca24-26/ocr/dataset/Word_Level_Hindi_Training_Set/Word_Level_Training_Set"
TRAIN_TXT = os.path.join(ROOT_DIR, "train.txt")
CHECKPOINT_PATH = "/home/mca24-26/ocr/hindi_xlm.pth"   # adjust if needed

IMG_HEIGHT = 64
IMG_WIDTH  = 256
MAX_SEQ_LEN = 40
BEAM_SIZE = 5            # beam width for testing
D_MODEL = 768
BATCH_SIZE = 32          # can be larger for inference
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_WORKERS = 4
SEED = 42

# ============================================================
# SEED
# ============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed(SEED)

# ============================================================
# VOCABULARY & TOKENIZER (same as training)
# ============================================================

HINDI_CHARS = (
    "अआइईउऊऋएऐओऔ"
    "कखगघङचछजझञ"
    "टठडढणतथदधन"
    "पफबभमयरलव"
    "शषसह"
    "ािीुूृेैोौंःॅ्"
    "०१२३४५६७८९"
    "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ"
    ".,!?()[]{}:;-_'/\\&%$#@+*= "
)

SPECIAL_TOKENS = ["<PAD>", "<SOS>", "<EOS>", "<UNK>"]
ALL_TOKENS = SPECIAL_TOKENS + list(HINDI_CHARS)
char2idx = {c: i for i, c in enumerate(ALL_TOKENS)}
idx2char = {i: c for c, i in char2idx.items()}

VOCAB_SIZE = len(ALL_TOKENS)
PAD_IDX = char2idx["<PAD>"]
SOS_IDX = char2idx["<SOS>"]
EOS_IDX = char2idx["<EOS>"]
UNK_IDX = char2idx["<UNK>"]

class HindiCustomTokenizer:
    def __init__(self):
        self.vocab_size = VOCAB_SIZE
        self.pad_token_id = PAD_IDX
        self.bos_token_id = SOS_IDX
        self.eos_token_id = EOS_IDX
        self.unk_token_id = UNK_IDX

    def decode(self, tokens, skip_special_tokens=True):
        chars = []
        for t in tokens:
            t = int(t)
            if t == self.eos_token_id:
                break
            if skip_special_tokens and t in [self.pad_token_id, self.bos_token_id]:
                continue
            chars.append(idx2char.get(t, ""))
        return "".join(chars)

tokenizer = HindiCustomTokenizer()

# ============================================================
# IMAGE PREPROCESSING (same as training)
# ============================================================

def preprocess_image(img_path):
    img = cv2.imread(img_path)
    if img is None:
        return torch.ones((3, IMG_HEIGHT, IMG_WIDTH), dtype=torch.float32) * -1.0
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
    edges = cv2.Canny(thresh, 50, 150, apertureSize=3)
    lines = cv2.HoughLinesP(edges, 1, np.pi/180, threshold=40, minLineLength=30, maxLineGap=5)
    angle = 0.0
    if lines is not None:
        angles = []
        for line in lines:
            x1, y1, x2, y2 = line[0]
            if x2 != x1:
                angles.append(np.arctan2(y2 - y1, x2 - x1) * 180 / np.pi)
        if angles:
            angle = np.median(angles)
    if abs(angle) > 20:
        angle = 0.0
    (h, w) = img.shape[:2]
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    deskewed = cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)
    
    h, w = deskewed.shape[:2]
    scale = IMG_HEIGHT / max(h, 1)
    new_w = int(w * scale)
    resized = cv2.resize(deskewed, (new_w, IMG_HEIGHT))
    if new_w < IMG_WIDTH:
        pad = np.ones((IMG_HEIGHT, IMG_WIDTH - new_w, 3), dtype=np.uint8) * 255
        resized = np.concatenate([resized, pad], axis=1)
    else:
        resized = cv2.resize(resized, (IMG_WIDTH, IMG_HEIGHT))
    
    img_tensor = resized.astype(np.float32) / 255.0
    img_tensor = (img_tensor - 0.5) / 0.5
    img_tensor = np.transpose(img_tensor, (2, 0, 1))
    return torch.tensor(img_tensor, dtype=torch.float32)

# ============================================================
# DATASET (test split only)
# ============================================================

class HindiWordDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        img_path, text = self.samples[idx]
        image = preprocess_image(img_path)
        return image, text

def load_data():
    samples = []
    with open(TRAIN_TXT, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split("\t")
            if len(parts) != 2:
                continue
            rel_path, text = parts
            text = unicodedata.normalize("NFC", text.strip())
            img_path = os.path.join(ROOT_DIR, rel_path)
            if os.path.exists(img_path):
                samples.append((img_path, text))
    print(f"Total samples: {len(samples)}")
    # Use the same split as training (test set is 7.5% of total: 0.15 * 0.5)
    _, temp = train_test_split(samples, test_size=0.15, random_state=SEED)
    _, test_samples = train_test_split(temp, test_size=0.5, random_state=SEED)
    print(f"Test samples: {len(test_samples)}")
    return test_samples

test_samples = load_data()
test_loader = DataLoader(
    HindiWordDataset(test_samples),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

# ============================================================
# MODEL ARCHITECTURE (same as training)
# ============================================================

class LocalizationNetwork(nn.Module):
    def __init__(self, num_control_points=16):
        super().__init__()
        self.num_control_points = num_control_points
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(True), nn.AdaptiveAvgPool2d((4, 4))
        )
        self.fc = nn.Sequential(
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(True),
            nn.Linear(256, num_control_points * 2)
        )
        self.fc[-1].weight.data.zero_()
        self.fc[-1].bias.data.zero_()

    def forward(self, x):
        B = x.size(0)
        pts = self.fc(self.conv(x).view(B, -1))
        return pts.view(B, self.num_control_points, 2)

class TPSSpatialTransformer(nn.Module):
    def __init__(self, num_control_points=16):
        super().__init__()
        self.num_control_points = num_control_points
        self.localization = LocalizationNetwork(num_control_points)
        r1 = torch.linspace(-0.9, 0.9, num_control_points // 2)
        r2 = torch.tensor([-0.7, 0.7])
        grid_Y, grid_X = torch.meshgrid(r2, r1, indexing='ij')
        target_control_points = torch.stack([grid_X.flatten(), grid_Y.flatten()], dim=1)
        self.register_buffer('target_control_points', target_control_points)

    def _compute_partial_solutions(self, X, Y):
        N = X.size(0)
        M = Y.size(0)
        X_tiled = X.unsqueeze(1).repeat(1, M, 1)
        Y_tiled = Y.unsqueeze(0).repeat(N, 1, 1)
        dist_sq = torch.sum((X_tiled - Y_tiled) ** 2, dim=2)
        dist_sq[dist_sq == 0] = 1.0
        return dist_sq * torch.log(dist_sq)

    def forward(self, x):
        B, C, H, W = x.size()
        source_control_points = self.localization(x) + self.target_control_points.unsqueeze(0)
        N = self.num_control_points
        L = torch.zeros(B, N + 3, N + 3, device=x.device)
        K = self._compute_partial_solutions(self.target_control_points, self.target_control_points)
        L[:, :N, :N] = K.unsqueeze(0)
        P = torch.cat([torch.ones(N, 1, device=x.device), self.target_control_points], dim=1)
        L[:, :N, N:] = P.unsqueeze(0)
        L[:, N:, :N] = P.t().unsqueeze(0)
        Y = torch.zeros(B, N + 3, 2, device=x.device)
        Y[:, :N, :] = source_control_points
        W_coef = torch.linalg.solve(L, Y)
        grid_Y, grid_X = torch.meshgrid(torch.linspace(-1, 1, H), torch.linspace(-1, 1, W), indexing='ij')
        eval_points = torch.stack([grid_X.flatten(), grid_Y.flatten()], dim=1).to(x.device)
        K_eval = self._compute_partial_solutions(eval_points, self.target_control_points)
        P_eval = torch.cat([torch.ones(H * W, 1, device=x.device), eval_points], dim=1)
        transformed_points = torch.matmul(K_eval, W_coef[:, :N, :]) + torch.matmul(P_eval, W_coef[:, N:, :])
        grid = transformed_points.view(B, H, W, 2)
        return F.grid_sample(x, grid, align_corners=False, padding_mode='border')

class SwinEncoder(nn.Module):
    def __init__(self, d_model=768):
        super().__init__()
        self.swin = timm.create_model(
            "swin_base_patch4_window7_224",
            pretrained=True,
            features_only=True,
            img_size=(IMG_HEIGHT, IMG_WIDTH),
            strict_img_size=False,
        )
        self.proj = nn.Linear(512, d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        features = self.swin(x)
        feat = features[-2]
        B, H, W, C = feat.shape
        feat = feat.view(B, H*W, C)
        return self.norm(self.proj(feat))

class XLMRobertaDecoder(nn.Module):
    def __init__(self, vocab_size, d_model=768):
        super().__init__()
        xlm_roberta = XLMRobertaModel.from_pretrained("xlm-roberta-base")
        config = XLMRobertaConfig.from_pretrained("xlm-roberta-base")
        config.is_decoder = True
        config.add_cross_attention = True
        config.vocab_size = vocab_size
        self.decoder = XLMRobertaForCausalLM(config)
        emb_state_dict = xlm_roberta.embeddings.state_dict()
        if "word_embeddings.weight" in emb_state_dict:
            emb_state_dict["word_embeddings.weight"] = emb_state_dict["word_embeddings.weight"][:vocab_size]
        self.decoder.roberta.embeddings.load_state_dict(emb_state_dict, strict=False)
        for i, layer in enumerate(self.decoder.roberta.encoder.layer[:6]):
            layer.load_state_dict(xlm_roberta.encoder.layer[i].state_dict(), strict=False)
        del xlm_roberta

    def forward(self, input_ids, encoder_hidden_states=None, labels=None):
        return self.decoder(input_ids=input_ids, encoder_hidden_states=encoder_hidden_states, labels=labels)

class CompleteHTRPipeline(nn.Module):
    def __init__(self, vocab_size, d_model=768, num_control_points=16):
        super().__init__()
        self.tps_stn = TPSSpatialTransformer(num_control_points)
        self.swin_encoder = SwinEncoder(d_model=d_model)
        self.bilstm = nn.LSTM(
            input_size=d_model, hidden_size=d_model//2, num_layers=2,
            batch_first=True, bidirectional=True, dropout=0.3
        )
        self.roberta_decoder = XLMRobertaDecoder(vocab_size=vocab_size, d_model=d_model)

    def _extract_visual_memory(self, images):
        rectified = self.tps_stn(images)
        swin_feat = self.swin_encoder(rectified)
        memory, _ = self.bilstm(swin_feat)
        return memory.contiguous()

    def forward(self, images, target_ids, criterion=None):
        memory = self._extract_visual_memory(images)
        dec_input = target_ids[:, :-1].clone()
        labels = target_ids[:, 1:].clone()
        dec_input[dec_input == -100] = PAD_IDX
        outputs = self.roberta_decoder(input_ids=dec_input, encoder_hidden_states=memory)
        logits = outputs.logits
        if criterion is not None:
            return criterion(logits.reshape(-1, logits.size(-1)), labels.reshape(-1))
        return F.cross_entropy(logits.reshape(-1, logits.size(-1)), labels.reshape(-1), ignore_index=-100)

    @torch.no_grad()
    def generate_greedy(self, images, max_length=MAX_SEQ_LEN, bos_token_id=SOS_IDX, eos_token_id=EOS_IDX):
        device = images.device
        B = images.size(0)
        memory = self._extract_visual_memory(images)
        generated = torch.full((B, 1), bos_token_id, dtype=torch.long, device=device)
        finished = torch.zeros(B, dtype=torch.bool, device=device)
        for _ in range(max_length - 1):
            outputs = self.roberta_decoder(input_ids=generated, encoder_hidden_states=memory)
            next_tokens = outputs.logits[:, -1, :].argmax(dim=-1, keepdim=True)
            next_tokens = next_tokens.masked_fill(finished.unsqueeze(1), eos_token_id)
            generated = torch.cat([generated, next_tokens], dim=1)
            finished |= (next_tokens.squeeze(1) == eos_token_id)
            if finished.all():
                break
        return generated[:, 1:]   # strip BOS

    @torch.no_grad()
    def generate_beam_search(self, images, max_length=MAX_SEQ_LEN, bos_token_id=SOS_IDX,
                             eos_token_id=EOS_IDX, beam_size=BEAM_SIZE):
        """Full beam search implementation."""
        device = images.device
        B = images.size(0)
        memory = self._extract_visual_memory(images)

        all_results = []
        for b in range(B):
            mem = memory[b:b+1]
            beams = [(0.0, [bos_token_id])]
            completed = []

            for _ in range(max_length - 1):
                candidates = []
                for score, tokens in beams:
                    if tokens[-1] == eos_token_id:
                        completed.append((score, tokens))
                        continue
                    inp = torch.tensor([tokens], dtype=torch.long, device=device)
                    out = self.roberta_decoder(input_ids=inp, encoder_hidden_states=mem)
                    log_prob = F.log_softmax(out.logits[0, -1], dim=-1)
                    top_lp, top_id = log_prob.topk(beam_size)
                    for lp, tid in zip(top_lp.tolist(), top_id.tolist()):
                        candidates.append((score + lp, tokens + [tid]))
                if not candidates:
                    break
                candidates.sort(key=lambda x: x[0], reverse=True)
                beams = []
                for score, tokens in candidates[:beam_size * 2]:
                    if tokens[-1] == eos_token_id:
                        completed.append((score, tokens))
                    else:
                        beams.append((score, tokens))
                    if len(beams) == beam_size:
                        break
                if not beams:
                    break
            all_beams = completed if completed else beams
            _, best_tokens = max(all_beams, key=lambda x: x[0])
            result = torch.tensor(best_tokens[1:], dtype=torch.long, device=device)
            all_results.append(result)

        max_len = max(r.size(0) for r in all_results)
        padded = torch.full((B, max_len), eos_token_id, dtype=torch.long, device=device)
        for i, r in enumerate(all_results):
            padded[i, :r.size(0)] = r
        return padded

# ============================================================
# AGENTIC VERIFICATION (lexicon from training)
# ============================================================

class AgenticVerificationModule:
    def __init__(self, train_samples):
        self.lexicon = defaultdict(int)
        for _, text in train_samples:
            for word in text.split():
                clean = word.strip(".,!?()[]{}:;\"'").lower()
                if clean:
                    self.lexicon[clean] += 1
        self.freq_max = max(self.lexicon.values()) if self.lexicon else 1
        print(f"Lexicon built: {len(self.lexicon)} unique words")

    def verify_and_correct(self, text_output, confidence=None, confidence_threshold=0.85):
        cleaned = text_output.strip().lower()
        if (not cleaned or cleaned in self.lexicon or len(cleaned) <= 2
                or any(c.isdigit() for c in cleaned)):
            return text_output
        if confidence is not None and confidence >= confidence_threshold:
            return text_output

        best_candidate = None
        best_score = -float('inf')
        target_len = len(cleaned)

        for word, freq in self.lexicon.items():
            if abs(len(word) - target_len) > 2:
                continue
            dist = Levenshtein.distance(cleaned, word)
            if dist > 2:
                continue
            freq_score = freq / self.freq_max
            score = freq_score - (dist * 1.2)
            if score > best_score:
                best_score = score
                best_candidate = word

        if best_candidate is None:
            return text_output
        if text_output.isupper():
            return best_candidate.upper()
        elif len(text_output) > 0 and text_output[0].isupper():
            return best_candidate.capitalize()
        return best_candidate

# ============================================================
# METRICS
# ============================================================

def char_accuracy(preds, labels):
    correct = 0
    total = 0
    for p, l in zip(preds, labels):
        n = min(len(p), len(l))
        for i in range(n):
            if p[i] == l[i]:
                correct += 1
        total += max(len(p), len(l))
    return 100.0 * correct / max(total, 1)

def calculate_metrics(preds, targets):
    cer = char_accuracy(preds, targets)
    wer_val = wer(targets, preds) * 100
    return {"CER": cer, "WER": wer_val}

# ============================================================
# MAIN TEST
# ============================================================

def main(use_beam_search=True, beam_size=BEAM_SIZE):
    print("=" * 60)
    print("TESTING HINDI HTR – XLM‑RoBERTa PIPELINE")
    print("=" * 60)

    # Load model
    model = CompleteHTRPipeline(vocab_size=VOCAB_SIZE).to(DEVICE)
    if not os.path.exists(CHECKPOINT_PATH):
        print(f"Checkpoint not found: {CHECKPOINT_PATH}")
        return
    ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f"Loaded best model from epoch {ckpt.get('epoch', '?')} "
          f"(WER: {ckpt.get('best_wer', '?')}%)")

    # Agentic verifier (need training samples to build lexicon)
    # We'll load training samples again (or you can pass pre-built lexicon)
    train_samples = []
    with open(TRAIN_TXT, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split("\t")
            if len(parts) == 2:
                train_samples.append(("", parts[1].strip()))
    agent_verifier = AgenticVerificationModule(train_samples)

    model.eval()
    all_preds_raw = []
    all_preds_verified = []
    all_labels = []

    inference_mode = "beam search" if use_beam_search else "greedy"
    print(f"\nRunning inference with {inference_mode} (beam_size={beam_size if use_beam_search else 1})...")

    with torch.no_grad():
        for images, texts in tqdm(test_loader, desc="Inference"):
            images = images.to(DEVICE)
            if use_beam_search:
                tokens = model.generate_beam_search(images, beam_size=beam_size)
            else:
                tokens = model.generate_greedy(images)

            for i in range(len(tokens)):
                raw = tokenizer.decode(tokens[i])
                verified = agent_verifier.verify_and_correct(raw)
                all_preds_raw.append(raw)
                all_preds_verified.append(verified)
                all_labels.append(texts[i])

    # Metrics without verification
    metrics_raw = calculate_metrics(all_preds_raw, all_labels)
    # Metrics with verification
    metrics_verified = calculate_metrics(all_preds_verified, all_labels)

    print("\n" + "=" * 60)
    print("TEST RESULTS")
    print("=" * 60)
    print(f"{'Metric':<20} {'Raw':>12} {'Verified':>12} {'Improvement':>12}")
    print("-" * 56)
    cer_imp = metrics_raw['CER'] - metrics_verified['CER']
    wer_imp = metrics_raw['WER'] - metrics_verified['WER']
    print(f"{'CER (%)':<20} {metrics_raw['CER']:>12.2f} {metrics_verified['CER']:>12.2f} {cer_imp:>+12.2f}")
    print(f"{'WER (%)':<20} {metrics_raw['WER']:>12.2f} {metrics_verified['WER']:>12.2f} {wer_imp:>+12.2f}")
    print("=" * 60)

    print("\nSample predictions (first 10):")
    for i in range(min(10, len(all_labels))):
        print(f"GT : {all_labels[i]}")
        print(f"PR : {all_preds_verified[i]}")
        print("-" * 40)

if __name__ == "__main__":
    # Run with beam search (set to False for greedy)
    main(use_beam_search=True, beam_size=5)

Total samples: 85585
Test samples: 6419
TESTING HINDI HTR – XLM‑RoBERTa PIPELINE


/usr/local/lib/python3.8/dist-packages/timm/layers/interpolate.py:47: UserWarning: torch.searchsorted(): input value tensor is non-contiguous, this will lower the performance due to extra data copy when converting non-contiguous tensor to contiguous, please use contiguous input value tensor if possible. This message will only appear once per program. (Triggered internally at ../aten/src/ATen/native/BucketizationUtils.h:32.)
  idx_right = torch.bucketize(x, p)
/usr/local/lib/python3.8/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/tmp/ipykernel_3057956/3465207824.py:477: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execu

Loaded best model from epoch 49 (WER: 5.0553%)
Lexicon built: 7605 unique words

Running inference with beam search (beam_size=5)...


Inference:   0%|                                                  | 0/201 [00:00<?, ?it/s]We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.
